[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C44_Adversarial_Security_Course/03_extraction_inversion/03_extraction_inversion.ipynb)

# 03 · 模型窃取与反演（用 numpy）

目标：在自建玩具目标模型（2 层 MLP）上从零实现 **功能窃取**（查询训替身）、**模型反演**（梯度上升重建输入），并用 **降信息量 / 水印** 防御。

> **防御视角**：在自己的玩具模型上演示，目的是理解 API 会泄露什么、如何设防。不涉及他人模型。

> 用 MLP 而非线性模型：线性模型的决策函数是全局的，**水印无法局部化**（会随窃取一起被复制）；MLP 的非线性使「埋在某个偏僻区域的水印」能被目标记住、却不被替身复制——这正是水印取证的前提。

路线：黑盒 MLP 目标 → 随机查询训替身 → 保真度 vs 查询预算 → 降信息量防御(top-1) → 模型反演 → 水印取证 → ✏️ 练习 → 📖 答案 → 🧪 查询效率胶囊。

## 1 · 黑盒目标模型（我们的「API」）

训一个**多类** 2 层 MLP 当目标。攻击者**只能查询**：给输入，拿回置信度（或只拿 top-1 标签）。我们能看到它的参数只是为了验证，攻击者看不到。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def make_multiclass(n=900, d=4, k=3, seed=0, spread=2.0):
    r = np.random.default_rng(seed)
    centers = r.standard_normal((k, d)) * spread
    X = []; y = []
    for c_ in range(k):
        X.append(r.standard_normal((n//k, d)) + centers[c_]); y += [c_]*(n//k)
    X = np.vstack(X); y = np.array(y)
    p = r.permutation(len(y)); return X[p], y[p], centers

def softmax(Z):
    Z = Z - Z.max(1, keepdims=True); E = np.exp(Z); return E / E.sum(1, keepdims=True)

class MLPClf:
    '''多类 2 层 MLP：tanh 隐层 + softmax 输出。可算参数梯度(训练/拟合)与输入梯度(反演)。'''
    def __init__(s, d=4, h=16, k=3, seed=0):
        r = np.random.default_rng(seed)
        s.W1 = r.standard_normal((d,h))*0.5; s.b1 = np.zeros(h)
        s.W2 = r.standard_normal((h,k))*0.5; s.b2 = np.zeros(k); s.k = k
    def forward(s, X):
        s.z1 = X@s.W1 + s.b1; s.a1 = np.tanh(s.z1)
        s.z2 = s.a1@s.W2 + s.b2; return softmax(s.z2)
    def proba(s, X): return s.forward(X)
    def predict(s, X): return s.proba(X).argmax(1)
    def grad_input_logprob(s, X, c):
        '''d log p_c / dx，逐样本(反演要用)。'''
        P = s.forward(X)
        dz2 = -P.copy(); dz2[:, c] += 1.0                 # d log p_c / dz2 = e_c - p
        dz1 = (dz2 @ s.W2.T) * (1 - np.tanh(s.z1)**2)
        return dz1 @ s.W1.T
    def fit_soft(s, X, Ysoft, lr=0.2, epochs=600, l2=1e-4):
        '''用软标签(置信度)拟合 —— 窃取替身要用。'''
        n = len(X)
        for _ in range(epochs):
            P = s.forward(X); dz2 = (P - Ysoft) / n
            dW2 = s.a1.T@dz2 + l2*s.W2; db2 = dz2.sum(0)
            dz1 = (dz2 @ s.W2.T) * (1 - np.tanh(s.z1)**2)
            dW1 = X.T@dz1 + l2*s.W1; db1 = dz1.sum(0)
            s.W2 -= lr*dW2; s.b2 -= lr*db2; s.W1 -= lr*dW1; s.b1 -= lr*db1
        return s
    def fit(s, X, y, lr=0.2, epochs=800, l2=1e-4):
        return s.fit_soft(X, np.eye(s.k)[y], lr, epochs, l2)

def acc(m, X, y): return float(np.mean(m.predict(X) == y))

X, y, centers = make_multiclass()
Xtr, ytr, Xte, yte = X[:600], y[:600], X[600:], y[600:]
victim = MLPClf().fit(Xtr, ytr)
print(f'目标(victim)模型测试精度={acc(victim,Xte,yte):.3f}')
assert acc(victim,Xte,yte) > 0.8
print('✅ 黑盒目标就绪（攻击者只能查询它）')

## 2 · 功能窃取：随机查询训替身

攻击者用随机点查询目标、拿回**软标签（置信度）**，用交叉熵拟合训出替身。验证替身与目标的**保真度**（预测一致率）高。

In [ ]:
def steal(victim, n_query, d=4, k=3, return_proba=True, seed=1, qscale=2.5):
    r = np.random.default_rng(seed)
    Xq = r.standard_normal((n_query, d)) * qscale       # 攻击者自造查询点
    if return_proba:
        Yq = victim.proba(Xq)                            # 软标签（信息多）
    else:
        Yq = np.eye(k)[victim.predict(Xq)]               # 只 top-1（信息少）
    return MLPClf(seed=2).fit_soft(Xq, Yq)

def fidelity(sur, victim, Xeval):
    return float(np.mean(sur.predict(Xeval) == victim.predict(Xeval)))

sur = steal(victim, n_query=2000, return_proba=True)
fid = fidelity(sur, victim, Xte)
print(f'替身保真度（与目标预测一致率）={fid:.3f}')
assert fid > 0.8, '足够查询应窃取出高保真替身'
print('✅ 窃取成功：仅凭查询返回训出功能相近的副本')

## 3 · 保真度 vs 查询预算

保真度随查询数上升并**饱和**。这条曲线是窃取的经济学，也是防御要对抗的对象。

In [ ]:
budgets = [50, 200, 500, 1500, 4000]
fids = []
for nq in budgets:
    f_ = fidelity(steal(victim, nq, return_proba=True), victim, Xte)
    fids.append(f_); print(f'  查询 {nq:>5} → 保真度 {f_:.3f}')
assert fids[-1] >= fids[0] - 1e-9, '保真度应随查询预算大体上升/饱和'
print('✅ 保真度随查询预算上升并饱和 —— 防御目标=抬高达到给定保真度所需查询数')

## 4 · 防御：降低输出信息量（只返回 top-1）

若 API 只返回 **top-1 标签** 而非完整置信度，每次查询泄露的信息变少，**相同查询预算下窃取保真度不升（通常下降）**。

In [ ]:
nq = 300
fid_proba = fidelity(steal(victim, nq, return_proba=True), victim, Xte)
fid_top1  = fidelity(steal(victim, nq, return_proba=False), victim, Xte)
print(f'相同 {nq} 次查询：返回完整置信度→保真度 {fid_proba:.3f} | 只返回 top-1→ {fid_top1:.3f}')
assert fid_top1 <= fid_proba + 1e-9, '降信息量应使窃取更难（保真度不升）'
print('✅ 降信息量防御：只回 top-1 抬高了窃取成本（攻击者失去富信息的置信度）')

## 5 · 模型反演：梯度上升重建输入

固定目标模型，对输入做**梯度上升最大化某类置信度**（+ 正则约束「像合理输入」），看重建结果是否逼近该类训练样本**中心**。

用反向传播算 $\nabla_x \log p_c(x)$（已在 `grad_input_logprob` 实现）。

In [ ]:
def invert_class(victim, target_c, d=4, steps=400, lr=0.5, reg=0.02, seed=0):
    r = np.random.default_rng(seed)
    x = r.standard_normal(d) * 0.1                 # 从近原点起
    for _ in range(steps):
        g = victim.grad_input_logprob(x[None], target_c)[0]
        x = x + lr * g - lr * reg * x              # 上升 + L2 正则(防跑飞)
    return x

for c_ in range(3):
    xrec = invert_class(victim, c_)
    nearest = np.argmin(np.linalg.norm(centers - xrec, axis=1))
    print(f'  反演类 {c_}: 重建点最近的类中心 = {nearest}  {"✓" if nearest==c_ else "✗"}')
    assert nearest == c_, '反演应重建出逼近目标类的输入'
print('✅ 模型反演：梯度上升爬出“模型心目中该类的样子”，逼近训练样本中心')

## 6 · 水印：窃取取证

给目标模型嵌入**触发式水印**：一簇藏在**偏僻区域**的触发点 → 一个指定的「暗号」类。攻击者的查询不会覆盖这个偏僻区域，**替身复制不出水印** → 可作所有权证据。（线性模型做不到这点，MLP 的非线性才能局部化水印。）

In [ ]:
WM_TARGET = 2
def make_watermark(d=4, m=30, seed=7, dist=12.0):
    r = np.random.default_rng(seed)
    secret = np.ones(d) / np.sqrt(d) * dist        # 远离正常数据的偏僻区域
    Xw = secret + 0.3 * r.standard_normal((m, d))  # 触发点簇
    return Xw, np.full(m, WM_TARGET), secret

Xw, yw, secret = make_watermark()
nat = np.argmin(np.linalg.norm(centers - secret, axis=1))   # 该区域“自然”应属哪类
wm_victim = MLPClf().fit(np.vstack([Xtr, Xw]), np.concatenate([ytr, yw]))
def wm_match(model): return float(np.mean(model.predict(Xw) == WM_TARGET))
print(f'水印区域自然类={nat}, 暗号类={WM_TARGET}')
print(f'带水印目标 在水印触发点的匹配率={wm_match(wm_victim):.3f} (高=记住了暗号)')
stolen = steal(wm_victim, 2500, return_proba=True)         # 攻击者窃取(不会查询偏僻区)
print(f'替身 在水印触发点的匹配率={wm_match(stolen):.3f} (低=没复制出暗号)')
assert wm_match(wm_victim) > wm_match(stolen) + 0.3, '替身复制不出水印 → 可作取证'
print('✅ 水印：原模型记得暗号、替身记不住 → 事后可证明谁是原作（取证/威慑）')

---
## ✏️ 练习区

### ✏️ 练习 1：用真实分布的查询更高效

上面用纯随机点查询。实现 `steal_indomain`：用**接近真实数据分布**的查询点（在类中心附近采样）训替身，验证相同预算下它的保真度**不低于**纯随机查询（更聪明的查询更高效）。

In [ ]:
def steal_indomain(victim, centers, n_query, d=4, k=3, seed=3):
    r = np.random.default_rng(seed)
    # TODO: 在各类中心附近采样 n_query 个查询点(更像真实数据)，查询victim软标签，fit_soft 训替身并返回
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
nq = 300
fid_rand = fidelity(steal(victim, nq, return_proba=True), victim, Xte)
fid_dom  = fidelity(steal_indomain(victim, centers, nq), victim, Xte)
print(f'随机查询保真度={fid_rand:.3f} | 贴分布查询={fid_dom:.3f}')
assert fid_dom >= fid_rand - 0.05, '贴近数据分布的查询应不差于纯随机'
print('✅ 练习 1 通过：查询点选得好，窃取更高效')

### ✏️ 练习 2：保真度 vs 准确率的区别

实现 `fid_and_acc(nq)`：返回替身的 `(保真度, 真实标签准确率)`。理解窃取关心的是**保真度**（学目标，含其错误），与在真实标签上的准确率是两个目标。

In [ ]:
def fid_and_acc(nq=2000):
    # TODO: 训替身，返回 (fidelity(替身,目标,Xte), acc(替身,Xte,yte))
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
f_, a_ = fid_and_acc(2000)
print(f'替身: 对目标保真度={f_:.3f}  对真标签准确率={a_:.3f}')
assert 0 <= f_ <= 1 and 0 <= a_ <= 1
assert f_ > 0.8, '保真度应较高（成功窃取功能）'
print('✅ 练习 2 通过：保真度衡量“多像目标”，与“多准”是两回事')

### ✏️ 练习 3：量化降信息量的防御增益

实现 `defense_gain_topk(nq)`：返回 `(完整置信度时保真度, 只top-1时保真度)`，验证前者≥后者（降信息量确实增加窃取难度）。

In [ ]:
def defense_gain_topk(nq=300):
    # TODO: 同一预算下，分别用 return_proba=True/False 窃取，返回两个保真度
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
full_, top1_ = defense_gain_topk(300)
print(f'完整置信度→保真度={full_:.3f} | 只top-1→{top1_:.3f}')
assert top1_ <= full_ + 1e-9, '降信息量应使窃取不更容易'
print('✅ 练习 3 通过')

### ✏️ 练习 4：反演的正则强度

反演里正则 `reg` 防止重建点跑飞。实现 `invert_dist(c, reg)` 返回重建点到目标类中心的距离，验证**适度正则**比**过强正则**重建得更靠近中心（过强正则会把点拉回原点）。

In [ ]:
def invert_dist(target_c, reg):
    # TODO: 用 invert_class(..., reg=reg) 重建，返回到 centers[target_c] 的欧氏距离
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
d_mod  = invert_dist(0, reg=0.02)
d_over = invert_dist(0, reg=2.0)
print(f'适度正则距中心={d_mod:.3f} | 过强正则={d_over:.3f}')
assert d_mod <= d_over + 1e-6, '适度正则应让反演更靠近中心(过强正则把点拉回原点)'
print('✅ 练习 4 通过')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def steal_indomain(victim, centers, n_query, d=4, k=3, seed=3):
    r = np.random.default_rng(seed)
    per = n_query // len(centers)
    Xq = np.vstack([r.standard_normal((per, d)) + centers[c_] for c_ in range(len(centers))])
    Yq = victim.proba(Xq)
    return MLPClf(seed=2).fit_soft(Xq, Yq)

In [ ]:
# 练习 2 参考答案
def fid_and_acc(nq=2000):
    s = steal(victim, nq, return_proba=True)
    return fidelity(s, victim, Xte), acc(s, Xte, yte)

In [ ]:
# 练习 3 参考答案
def defense_gain_topk(nq=300):
    full = fidelity(steal(victim, nq, return_proba=True), victim, Xte)
    top1 = fidelity(steal(victim, nq, return_proba=False), victim, Xte)
    return full, top1

In [ ]:
# 练习 4 参考答案
def invert_dist(target_c, reg):
    x = invert_class(victim, target_c, reg=reg)
    return float(np.linalg.norm(x - centers[target_c]))

---
## 🧪 真实数据胶囊：窃取的查询效率（主动 vs 随机）

复现 Papernot 2017 的直觉（玩具版）：**在决策边界附近查询**比随机查询更省。

边界附近的点信息量最大（一点点移动就换类），在那里多问，能用更少查询逼近目标边界。

### 胶囊练习：边界附近的主动查询

实现 `boundary_queries`：给当前替身，找出它**预测最不确定**（top-2 概率最接近）的查询点，用这些点查询目标来扩充训练集。这是主动学习/雅可比增广的核心直觉。

In [ ]:
def boundary_queries(surrogate, pool, n_pick):
    '''从候选池 pool 里挑替身最不确定的 n_pick 个点（top1 与 top2 概率差最小）。'''
    P = surrogate.proba(pool)
    # TODO: 算每个点的 top1-top2 概率差(margin)，返回 margin 最小的 n_pick 个点的下标
    raise NotImplementedError

In [ ]:
# —— 胶囊自测 ——（先做 TODO）
r = np.random.default_rng(9)
pool = r.standard_normal((3000, 4)) * 2.5
init = 100
sur0 = steal(victim, init, return_proba=True)
idx = boundary_queries(sur0, pool, 200)
Xq = np.vstack([r.standard_normal((init,4))*2.5, pool[idx]])
Yq = victim.proba(Xq)
sur_active = MLPClf(seed=2).fit_soft(Xq, Yq)
fid_active = fidelity(sur_active, victim, Xte)
fid_rand_same = fidelity(steal(victim, init+200, return_proba=True), victim, Xte)
print(f'主动(边界)查询 {init+200} 次 → 保真度 {fid_active:.3f}')
print(f'纯随机    查询 {init+200} 次 → 保真度 {fid_rand_same:.3f}')
assert idx.shape[0] == 200
print('✅ 胶囊通过：边界附近查询信息量大 → 同等预算下窃取更高效（防御要对这类查询模式做异常检测）')

In [ ]:
# 📖 胶囊参考答案
def boundary_queries(surrogate, pool, n_pick):
    P = surrogate.proba(pool)
    Ps = np.sort(P, axis=1)
    margin = Ps[:, -1] - Ps[:, -2]            # top1 - top2
    return np.argsort(margin)[:n_pick]

### 小结
- **模型窃取**：黑盒查询 → 收集软标签 → 训替身；保真度随**查询预算**上升并饱和。
- **保真度 vs 准确率**：窃取追求“多像目标”（含其错误），≠ “多准”。
- **模型反演**：梯度上升最大化某类置信度 → 重建逼近该类的输入（记忆=可还原）。
- 防御：**限流 + 降信息量(top-1/加噪) + DP**（预防/缓解）+ **水印**（取证/威慑），分层叠加。

下一站：**模块 04 · 成员推断与隐私** —— 连“某条数据是否被训练过”都能被问出来。